# 02 — Train Multi-Branch KoVAE

This is now the main KoVAE training notebook.

The implementation uses the stronger conditional design internally:

```text
activity embeddings
subject embeddings
activity classifier loss
activity-conditioned Koopman matrices
```

But the approach is named simply:

```text
kovae
```

because the old KoVAE outputs were deleted and this is now the main/final KoVAE method.

Main saved outputs:

```text
configs/kovae_config.json
models/checkpoints/kovae_best.pt
models/checkpoints/kovae_last.pt
results/kovae/
figures/kovae/
logs/kovae_training.log
```


In [1]:

# ============================================================
# 02_train_conditional_multibranch_kovae.py
#
# Train a multi-branch KoVAE on native-rate PPG-DaLiA windows.
#
# Inputs:
#   data/processed/native_rates/all_X_acc_32hz.npy
#   data/processed/native_rates/all_X_bvp_64hz.npy
#   data/processed/native_rates/all_X_slow_4hz.npy
#   data/processed/native_rates/all_y.npy
#   data/processed/native_rates/all_subject.npy
#
# Outputs:
#   configs/kovae_config.json
#   models/checkpoints/kovae_best.pt
#   models/checkpoints/kovae_last.pt
#   results/kovae/training_history.csv
#   results/kovae/final_metrics.json
#   results/kovae/activity_mapping.json
#   results/kovae/subject_mapping.json
#   figures/kovae/training_loss_curve.png
#   figures/kovae/reconstruction_examples.png
#   figures/kovae/activity_koopman_eigenvalues.png
#   logs/kovae_training.log
# ============================================================

from __future__ import annotations

import json
import logging
import math
import random
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


# ============================================================
# Configuration
# ============================================================

@dataclass
class KoVAEConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"
    processed_subdir: str = "data/processed/native_rates"

    # Separate experiment name so the conditional model does not mix with
    # the earlier KoVAE checkpoints/results.
    experiment_name: str = "kovae"

    random_seed: int = 42

    train_subjects: List[str] = field(default_factory=lambda: [
        "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
    ])
    val_subjects: List[str] = field(default_factory=lambda: ["S14", "S15"])
    test_subjects: List[str] = field(default_factory=lambda: ["S7", "S8", "S10"])

    acc_hz: int = 32
    bvp_hz: int = 64
    slow_hz: int = 4

    acc_len: int = 256
    bvp_len: int = 512
    slow_len: int = 32

    acc_channels: int = 3
    bvp_channels: int = 1
    slow_channels: int = 2

    latent_steps: int = 64
    branch_hidden_dim: int = 64
    fusion_hidden_dim: int = 128
    latent_dim: int = 32

    activity_embedding_dim: int = 16
    use_subject_condition: bool = True
    subject_embedding_dim: int = 16
    subject_dropout_prob: float = 0.20

    # Strong conditioning additions:
    # 1) latent activity classifier keeps activity information in z.
    # 2) one learned Koopman transition matrix per activity.
    activity_classifier_weight: float = 0.20
    use_activity_conditioned_koopman: bool = True
    activity_koopman_noise_std: float = 0.01
    activity_koopman_identity_regularization: float = 1e-5

    dropout: float = 0.10

    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 15
    gradient_clip_norm: float = 1.0

    use_weighted_sampler: bool = True
    use_amp: bool = True

    reconstruction_weight_bvp: float = 1.0
    reconstruction_weight_acc: float = 1.0
    reconstruction_weight_slow: float = 1.0

    alpha_koopman: float = 0.10
    beta_kl: float = 1e-3
    koopman_ridge: float = 1e-3

    max_reconstruction_examples: int = 3


CONFIG = KoVAEConfig()


# ============================================================
# Paths, logging, reproducibility
# ============================================================

def get_project_paths(config: KoVAEConfig) -> Dict[str, Path]:
    root = Path(config.project_root)
    experiment = config.experiment_name

    return {
        "project_root": root,
        "processed": root / config.processed_subdir,
        "configs": root / "configs",
        "figures": root / "figures" / experiment,
        "logs": root / "logs",
        "models": root / "models",
        "checkpoints": root / "models" / "checkpoints",
        "generators": root / "models" / "generators",
        "results": root / "results" / experiment,
    }


def create_dirs(paths: Dict[str, Path]) -> None:
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_logging(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("kovae_training")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    stream_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    return logger


def save_json(data: dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# Data loading and splitting
# ============================================================

def load_native_rate_arrays(processed_dir: Path) -> Dict[str, np.ndarray]:
    required_files = {
        "X_acc": "all_X_acc_32hz.npy",
        "X_bvp": "all_X_bvp_64hz.npy",
        "X_slow": "all_X_slow_4hz.npy",
        "y": "all_y.npy",
        "subjects": "all_subject.npy",
    }

    arrays = {}
    missing = []

    for key, filename in required_files.items():
        path = processed_dir / filename
        if not path.exists():
            missing.append(str(path))
        else:
            if key == "subjects":
                arrays[key] = np.load(path, allow_pickle=True).astype(str)
            else:
                arrays[key] = np.load(path)

    if missing:
        raise FileNotFoundError(
            "Missing required preprocessing files:\n" + "\n".join(missing)
        )

    return arrays


def validate_arrays(arrays: Dict[str, np.ndarray], config: KoVAEConfig) -> None:
    X_acc = arrays["X_acc"]
    X_bvp = arrays["X_bvp"]
    X_slow = arrays["X_slow"]
    y = arrays["y"]
    subjects = arrays["subjects"]

    n = len(y)

    if X_acc.shape != (n, config.acc_len, config.acc_channels):
        raise ValueError(f"Expected ACC shape {(n, config.acc_len, config.acc_channels)}, got {X_acc.shape}")

    if X_bvp.shape != (n, config.bvp_len, config.bvp_channels):
        raise ValueError(f"Expected BVP shape {(n, config.bvp_len, config.bvp_channels)}, got {X_bvp.shape}")

    if X_slow.shape != (n, config.slow_len, config.slow_channels):
        raise ValueError(f"Expected SLOW shape {(n, config.slow_len, config.slow_channels)}, got {X_slow.shape}")

    if len(subjects) != n:
        raise ValueError(f"subjects length {len(subjects)} does not match y length {n}")

    for name in ["X_acc", "X_bvp", "X_slow"]:
        if not np.all(np.isfinite(arrays[name])):
            raise ValueError(f"{name} contains NaN or infinite values.")


def build_indices_by_subject(subjects: np.ndarray, selected_subjects: List[str]) -> np.ndarray:
    mask = np.isin(subjects.astype(str), np.asarray(selected_subjects, dtype=str))
    return np.where(mask)[0].astype(np.int64)


def build_activity_mapping(y: np.ndarray) -> Dict[str, int]:
    unique_labels = sorted(int(label) for label in np.unique(y))
    return {str(label): idx for idx, label in enumerate(unique_labels)}


def build_subject_mapping(train_subjects: List[str]) -> Dict[str, int]:
    mapping = {"UNK": 0}
    for idx, subject in enumerate(sorted(train_subjects, key=lambda s: int(s[1:])), start=1):
        mapping[subject] = idx
    return mapping


def encode_activities(y: np.ndarray, mapping: Dict[str, int]) -> np.ndarray:
    encoded = []
    for label in y:
        key = str(int(label))
        if key not in mapping:
            raise KeyError(f"Activity label {key} not found in mapping.")
        encoded.append(mapping[key])
    return np.asarray(encoded, dtype=np.int64)


def encode_subjects(subjects: np.ndarray, mapping: Dict[str, int]) -> np.ndarray:
    return np.asarray([mapping.get(str(subject), mapping["UNK"]) for subject in subjects], dtype=np.int64)


# ============================================================
# Dataset
# ============================================================

class NativeRatePPGDataset(Dataset):
    def __init__(
        self,
        X_acc: np.ndarray,
        X_bvp: np.ndarray,
        X_slow: np.ndarray,
        y_encoded: np.ndarray,
        subject_encoded: np.ndarray,
        indices: np.ndarray,
    ) -> None:
        self.X_acc = X_acc
        self.X_bvp = X_bvp
        self.X_slow = X_slow
        self.y_encoded = y_encoded
        self.subject_encoded = subject_encoded
        self.indices = indices.astype(np.int64)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int) -> Dict[str, torch.Tensor]:
        idx = self.indices[item]

        return {
            "acc": torch.from_numpy(self.X_acc[idx]).float(),
            "bvp": torch.from_numpy(self.X_bvp[idx]).float(),
            "slow": torch.from_numpy(self.X_slow[idx]).float(),
            "activity": torch.tensor(self.y_encoded[idx], dtype=torch.long),
            "subject": torch.tensor(self.subject_encoded[idx], dtype=torch.long),
            "global_index": torch.tensor(idx, dtype=torch.long),
        }


def make_weighted_sampler(y_encoded: np.ndarray, indices: np.ndarray) -> WeightedRandomSampler:
    labels = y_encoded[indices]
    unique_labels, counts = np.unique(labels, return_counts=True)
    count_map = {label: count for label, count in zip(unique_labels, counts)}
    sample_weights = np.asarray([1.0 / count_map[label] for label in labels], dtype=np.float64)

    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).double(),
        num_samples=len(sample_weights),
        replacement=True,
    )


def create_dataloaders(
    arrays: Dict[str, np.ndarray],
    config: KoVAEConfig,
    logger: logging.Logger,
) -> Tuple[DataLoader, DataLoader, Dict[str, int], Dict[str, int], Dict[str, np.ndarray]]:
    y = arrays["y"]
    subjects = arrays["subjects"]

    activity_to_idx = build_activity_mapping(y)
    subject_to_idx = build_subject_mapping(config.train_subjects)

    y_encoded = encode_activities(y, activity_to_idx)
    subject_encoded = encode_subjects(subjects, subject_to_idx)

    train_indices = build_indices_by_subject(subjects, config.train_subjects)
    val_indices = build_indices_by_subject(subjects, config.val_subjects)
    test_indices = build_indices_by_subject(subjects, config.test_subjects)

    if len(train_indices) == 0:
        raise RuntimeError("No training windows found. Check train_subjects.")
    if len(val_indices) == 0:
        raise RuntimeError("No validation windows found. Check val_subjects.")

    train_dataset = NativeRatePPGDataset(
        arrays["X_acc"],
        arrays["X_bvp"],
        arrays["X_slow"],
        y_encoded,
        subject_encoded,
        train_indices,
    )

    val_dataset = NativeRatePPGDataset(
        arrays["X_acc"],
        arrays["X_bvp"],
        arrays["X_slow"],
        y_encoded,
        subject_encoded,
        val_indices,
    )

    if config.use_weighted_sampler:
        sampler = make_weighted_sampler(y_encoded, train_indices)
        train_loader = DataLoader(
            train_dataset,
            batch_size=config.batch_size,
            sampler=sampler,
            num_workers=config.num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=config.batch_size,
            shuffle=True,
            num_workers=config.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    logger.info("Activity mapping: %s", activity_to_idx)
    logger.info("Subject mapping: %s", subject_to_idx)
    logger.info("Train windows: %d", len(train_indices))
    logger.info("Val windows: %d", len(val_indices))
    logger.info("Test windows reserved for downstream only: %d", len(test_indices))

    split_indices = {
        "train_indices": train_indices,
        "val_indices": val_indices,
        "test_indices": test_indices,
    }

    return train_loader, val_loader, activity_to_idx, subject_to_idx, split_indices


# ============================================================
# Model modules
# ============================================================

class BranchEncoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int,
        latent_steps: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=7, padding=3),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(latent_steps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        h = self.net(x)
        h = self.pool(h)
        return h.transpose(1, 2)


class BranchDecoder(nn.Module):
    def __init__(
        self,
        latent_dim: int,
        condition_dim: int,
        hidden_dim: int,
        output_len: int,
        output_channels: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.output_len = output_len

        self.gru = nn.GRU(
            input_size=latent_dim + condition_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        self.conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z: torch.Tensor, condition_seq: torch.Tensor) -> torch.Tensor:
        h = torch.cat([z, condition_seq], dim=-1)
        h, _ = self.gru(h)
        h = h.transpose(1, 2)
        h = F.interpolate(h, size=self.output_len, mode="linear", align_corners=False)
        out = self.conv(h)
        return out.transpose(1, 2)


class MultiBranchKoVAE(nn.Module):
    """
    Conditional multi-branch KoVAE.

    Compared with the earlier model, this version makes activity conditioning
    stronger in two ways:

    1. Activity and subject embeddings are concatenated to the encoder fusion
       and to every decoder timestep.
    2. The latent representation is supervised with an auxiliary activity
       classifier.
    3. Koopman dynamics are activity-conditioned: each activity has its own
       learned linear latent transition matrix.
    """

    def __init__(
        self,
        config: KoVAEConfig,
        num_activities: int,
        num_subject_tokens: int,
    ) -> None:
        super().__init__()

        self.config = config
        self.num_activities = num_activities
        self.num_subject_tokens = num_subject_tokens

        self.bvp_encoder = BranchEncoder(
            in_channels=config.bvp_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.acc_encoder = BranchEncoder(
            in_channels=config.acc_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.slow_encoder = BranchEncoder(
            in_channels=config.slow_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )

        self.activity_embedding = nn.Embedding(num_activities, config.activity_embedding_dim)

        if config.use_subject_condition:
            self.subject_embedding = nn.Embedding(num_subject_tokens, config.subject_embedding_dim)
            condition_dim = config.activity_embedding_dim + config.subject_embedding_dim
        else:
            self.subject_embedding = None
            condition_dim = config.activity_embedding_dim

        self.condition_dim = condition_dim

        fusion_input_dim = 3 * config.branch_hidden_dim + condition_dim

        self.fusion_gru = nn.GRU(
            input_size=fusion_input_dim,
            hidden_size=config.fusion_hidden_dim,
            batch_first=True,
        )

        self.to_mu = nn.Linear(config.fusion_hidden_dim, config.latent_dim)
        self.to_logvar = nn.Linear(config.fusion_hidden_dim, config.latent_dim)

        self.activity_classifier = nn.Sequential(
            nn.LayerNorm(config.latent_dim),
            nn.Linear(config.latent_dim, config.fusion_hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.fusion_hidden_dim // 2, num_activities),
        )

        # One learned latent transition matrix per activity.
        self.activity_koopman = nn.Parameter(
            torch.empty(num_activities, config.latent_dim, config.latent_dim)
        )
        self.reset_activity_koopman_parameters()

        self.bvp_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.bvp_len,
            output_channels=config.bvp_channels,
            dropout=config.dropout,
        )
        self.acc_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.acc_len,
            output_channels=config.acc_channels,
            dropout=config.dropout,
        )
        self.slow_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.slow_len,
            output_channels=config.slow_channels,
            dropout=config.dropout,
        )

    def reset_activity_koopman_parameters(self) -> None:
        with torch.no_grad():
            eye = torch.eye(self.config.latent_dim)
            eye = eye[None, :, :].repeat(self.num_activities, 1, 1)

            noise = self.config.activity_koopman_noise_std * torch.randn_like(eye)
            self.activity_koopman.copy_(eye + noise)

    def make_condition(self, activity: torch.Tensor, subject: torch.Tensor) -> torch.Tensor:
        activity_emb = self.activity_embedding(activity)

        if self.config.use_subject_condition:
            subject_emb = self.subject_embedding(subject)
            condition = torch.cat([activity_emb, subject_emb], dim=-1)
        else:
            condition = activity_emb

        return condition

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(
        self,
        bvp: torch.Tensor,
        acc: torch.Tensor,
        slow: torch.Tensor,
        activity: torch.Tensor,
        subject: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        bvp_h = self.bvp_encoder(bvp)
        acc_h = self.acc_encoder(acc)
        slow_h = self.slow_encoder(slow)

        condition = self.make_condition(activity, subject)
        condition_seq = condition[:, None, :].repeat(1, self.config.latent_steps, 1)

        fused = torch.cat([bvp_h, acc_h, slow_h, condition_seq], dim=-1)
        fused_h, _ = self.fusion_gru(fused)

        mu = self.to_mu(fused_h)
        logvar = self.to_logvar(fused_h).clamp(min=-8.0, max=8.0)

        z = self.reparameterize(mu, logvar)

        recon_bvp = self.bvp_decoder(z, condition_seq)
        recon_acc = self.acc_decoder(z, condition_seq)
        recon_slow = self.slow_decoder(z, condition_seq)

        pooled_mu = mu.mean(dim=1)
        activity_logits = self.activity_classifier(pooled_mu)

        return {
            "recon_bvp": recon_bvp,
            "recon_acc": recon_acc,
            "recon_slow": recon_slow,
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "condition": condition,
            "activity_logits": activity_logits,
        }

    def get_activity_koopman_matrices(self) -> torch.Tensor:
        return self.activity_koopman


# ============================================================
# KoVAE loss
# ============================================================

def compute_global_koopman_diagnostic_matrix(
    latent_sequence: torch.Tensor,
    ridge: float,
) -> torch.Tensor:
    """
    Least-squares global Koopman matrix for diagnostics only.

    The training loss below uses learned activity-conditioned Koopman matrices.
    This global matrix is saved only for comparison with the earlier version.
    """

    if latent_sequence.shape[1] < 2:
        raise ValueError("Koopman diagnostic requires latent sequence length >= 2.")

    autocast_context = (
        torch.amp.autocast("cuda", enabled=False)
        if latent_sequence.is_cuda else nullcontext()
    )

    with autocast_context:
        z = latent_sequence.float()

        X = z[:, :-1, :].reshape(-1, z.shape[-1])
        Y = z[:, 1:, :].reshape(-1, z.shape[-1])

        dim = X.shape[-1]
        eye = torch.eye(dim, device=X.device, dtype=torch.float32)

        xtx = (X.T @ X) / max(1, X.shape[0])
        xty = (X.T @ Y) / max(1, X.shape[0])

        A = torch.linalg.solve(xtx + ridge * eye, xty)

    return A


def compute_activity_conditioned_koopman_loss(
    latent_sequence: torch.Tensor,
    activity: torch.Tensor,
    model: MultiBranchKoVAE,
    config: KoVAEConfig,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Activity-conditioned Koopman loss.

    For each sample, the activity label selects its own learned matrix A_y.
    The loss encourages:

        z_{t+1} ~= z_t A_y

    This is the main difference from the older global least-squares Koopman
    regularizer.
    """

    if latent_sequence.shape[1] < 2:
        raise ValueError("Koopman loss requires latent sequence length >= 2.")

    z = latent_sequence.float()

    z_t = z[:, :-1, :]
    z_next = z[:, 1:, :]

    A = model.get_activity_koopman_matrices()[activity].float()
    z_pred = torch.einsum("btd,bdh->bth", z_t, A)

    prediction_loss = F.mse_loss(z_pred, z_next)

    identity = torch.eye(
        model.config.latent_dim,
        device=z.device,
        dtype=torch.float32,
    )[None, :, :]

    identity_loss = F.mse_loss(A, identity.expand_as(A))

    total_koopman_loss = (
        prediction_loss
        + config.activity_koopman_identity_regularization * identity_loss
    )

    mean_A = A.mean(dim=0)

    return total_koopman_loss, prediction_loss, mean_A


def compute_kovae_loss(
    batch: Dict[str, torch.Tensor],
    outputs: Dict[str, torch.Tensor],
    model: MultiBranchKoVAE,
    config: KoVAEConfig,
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    bvp_loss = F.mse_loss(outputs["recon_bvp"], batch["bvp"])
    acc_loss = F.mse_loss(outputs["recon_acc"], batch["acc"])
    slow_loss = F.mse_loss(outputs["recon_slow"], batch["slow"])

    reconstruction_loss = (
        config.reconstruction_weight_bvp * bvp_loss
        + config.reconstruction_weight_acc * acc_loss
        + config.reconstruction_weight_slow * slow_loss
    )

    mu = outputs["mu"]
    logvar = outputs["logvar"]

    kl_loss = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())

    if config.use_activity_conditioned_koopman:
        koopman_loss, koopman_prediction_loss, A = compute_activity_conditioned_koopman_loss(
            latent_sequence=mu,
            activity=batch["activity"],
            model=model,
            config=config,
        )
    else:
        A = compute_global_koopman_diagnostic_matrix(
            latent_sequence=mu,
            ridge=config.koopman_ridge,
        )
        z = mu.float()
        X = z[:, :-1, :].reshape(-1, z.shape[-1])
        Y = z[:, 1:, :].reshape(-1, z.shape[-1])
        Y_pred = X @ A
        koopman_prediction_loss = F.mse_loss(Y_pred, Y)
        koopman_loss = koopman_prediction_loss

    activity_classifier_loss = F.cross_entropy(
        outputs["activity_logits"],
        batch["activity"],
    )

    activity_predictions = torch.argmax(outputs["activity_logits"], dim=1)
    activity_accuracy = (activity_predictions == batch["activity"]).float().mean()

    total_loss = (
        reconstruction_loss
        + config.beta_kl * kl_loss
        + config.alpha_koopman * koopman_loss
        + config.activity_classifier_weight * activity_classifier_loss
    )

    components = {
        "total_loss": total_loss.detach(),
        "reconstruction_loss": reconstruction_loss.detach(),
        "bvp_loss": bvp_loss.detach(),
        "acc_loss": acc_loss.detach(),
        "slow_loss": slow_loss.detach(),
        "kl_loss": kl_loss.detach(),
        "koopman_loss": koopman_loss.detach(),
        "koopman_prediction_loss": koopman_prediction_loss.detach(),
        "activity_classifier_loss": activity_classifier_loss.detach(),
        "activity_accuracy": activity_accuracy.detach(),
        "koopman_matrix": A.detach(),
    }

    return total_loss, components


# ============================================================
# Training and validation
# ============================================================

def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def apply_subject_dropout(
    subject: torch.Tensor,
    unk_index: int,
    dropout_prob: float,
) -> torch.Tensor:
    if dropout_prob <= 0:
        return subject

    mask = torch.rand_like(subject.float()) < dropout_prob
    dropped = subject.clone()
    dropped[mask] = unk_index
    return dropped


def run_one_epoch(
    model: MultiBranchKoVAE,
    loader: DataLoader,
    optimizer: Optional[torch.optim.Optimizer],
    scaler: Optional[torch.amp.GradScaler],
    config: KoVAEConfig,
    device: torch.device,
    train: bool,
    unk_subject_index: int,
) -> Tuple[Dict[str, float], Optional[np.ndarray]]:
    if train:
        model.train()
    else:
        model.eval()

    totals = {
        "total_loss": 0.0,
        "reconstruction_loss": 0.0,
        "bvp_loss": 0.0,
        "acc_loss": 0.0,
        "slow_loss": 0.0,
        "kl_loss": 0.0,
        "koopman_loss": 0.0,
        "koopman_prediction_loss": 0.0,
        "activity_classifier_loss": 0.0,
        "activity_accuracy": 0.0,
    }

    total_samples = 0
    last_A = None

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        batch_size = batch["bvp"].shape[0]
        total_samples += batch_size

        if train and config.use_subject_condition:
            subject_input = apply_subject_dropout(
                batch["subject"],
                unk_index=unk_subject_index,
                dropout_prob=config.subject_dropout_prob,
            )
        else:
            subject_input = batch["subject"]

        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)

        amp_enabled = config.use_amp and device.type == "cuda"

        autocast_context = (
            torch.amp.autocast("cuda", enabled=amp_enabled)
            if device.type == "cuda" else nullcontext()
        )

        with torch.set_grad_enabled(train):
            with autocast_context:
                outputs = model(
                    bvp=batch["bvp"],
                    acc=batch["acc"],
                    slow=batch["slow"],
                    activity=batch["activity"],
                    subject=subject_input,
                )
                loss, components = compute_kovae_loss(
                    batch=batch,
                    outputs=outputs,
                    model=model,
                    config=config,
                )

            if train:
                if scaler is not None and amp_enabled:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
                    optimizer.step()

        for key in totals:
            totals[key] += float(components[key].item()) * batch_size

        last_A = components["koopman_matrix"].detach().cpu().numpy()

    metrics = {key: value / max(1, total_samples) for key, value in totals.items()}
    return metrics, last_A


def save_checkpoint(
    path: Path,
    model: MultiBranchKoVAE,
    optimizer: torch.optim.Optimizer,
    config: KoVAEConfig,
    epoch: int,
    best_val_loss: float,
    activity_to_idx: Dict[str, int],
    subject_to_idx: Dict[str, int],
) -> None:
    checkpoint = {
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": asdict(config),
        "activity_to_idx": activity_to_idx,
        "subject_to_idx": subject_to_idx,
        "num_activities": len(activity_to_idx),
        "num_subject_tokens": len(subject_to_idx),
    }
    torch.save(checkpoint, path)


# ============================================================
# Plotting
# ============================================================

def plot_training_history(history_df: pd.DataFrame, figure_path: Path) -> None:
    plt.figure(figsize=(10, 6))
    plt.plot(history_df["epoch"], history_df["train_total_loss"], label="train total")
    plt.plot(history_df["epoch"], history_df["val_total_loss"], label="val total")
    plt.plot(history_df["epoch"], history_df["train_reconstruction_loss"], label="train reconstruction")
    plt.plot(history_df["epoch"], history_df["val_reconstruction_loss"], label="val reconstruction")
    if "train_activity_classifier_loss" in history_df.columns:
        plt.plot(history_df["epoch"], history_df["train_activity_classifier_loss"], label="train activity CE")
        plt.plot(history_df["epoch"], history_df["val_activity_classifier_loss"], label="val activity CE")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("KoVAE training history")
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_koopman_eigenvalues(A: np.ndarray, figure_path: Path) -> None:
    eigvals = np.linalg.eigvals(A)

    theta = np.linspace(0, 2 * np.pi, 400)
    unit_x = np.cos(theta)
    unit_y = np.sin(theta)

    plt.figure(figsize=(6, 6))
    plt.plot(unit_x, unit_y, linestyle="--", label="unit circle")
    plt.scatter(eigvals.real, eigvals.imag)
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlabel("Real")
    plt.ylabel("Imaginary")
    plt.title("Koopman matrix eigenvalues")
    plt.legend()
    plt.axis("equal")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_reconstruction_examples(
    model: MultiBranchKoVAE,
    loader: DataLoader,
    config: KoVAEConfig,
    device: torch.device,
    figure_path: Path,
) -> None:
    model.eval()

    batch = next(iter(loader))
    batch = move_batch_to_device(batch, device)

    with torch.no_grad():
        outputs = model(
            bvp=batch["bvp"],
            acc=batch["acc"],
            slow=batch["slow"],
            activity=batch["activity"],
            subject=batch["subject"],
        )

    num_examples = min(config.max_reconstruction_examples, batch["bvp"].shape[0])

    fig, axes = plt.subplots(num_examples, 3, figsize=(15, 4 * num_examples))

    if num_examples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_examples):
        t_bvp = np.arange(config.bvp_len) / config.bvp_hz
        axes[i, 0].plot(t_bvp, batch["bvp"][i, :, 0].detach().cpu().numpy(), label="real")
        axes[i, 0].plot(t_bvp, outputs["recon_bvp"][i, :, 0].detach().cpu().numpy(), label="recon")
        axes[i, 0].set_title("BVP 64 Hz")
        axes[i, 0].set_xlabel("Time (sec)")
        axes[i, 0].legend()

        t_acc = np.arange(config.acc_len) / config.acc_hz
        axes[i, 1].plot(t_acc, batch["acc"][i, :, 0].detach().cpu().numpy(), label="real ACC_x")
        axes[i, 1].plot(t_acc, outputs["recon_acc"][i, :, 0].detach().cpu().numpy(), label="recon ACC_x")
        axes[i, 1].set_title("ACC_x 32 Hz")
        axes[i, 1].set_xlabel("Time (sec)")
        axes[i, 1].legend()

        t_slow = np.arange(config.slow_len) / config.slow_hz
        axes[i, 2].plot(t_slow, batch["slow"][i, :, 0].detach().cpu().numpy(), label="real EDA")
        axes[i, 2].plot(t_slow, outputs["recon_slow"][i, :, 0].detach().cpu().numpy(), label="recon EDA")
        axes[i, 2].set_title("EDA 4 Hz")
        axes[i, 2].set_xlabel("Time (sec)")
        axes[i, 2].legend()

    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def save_activity_koopman_summary(
    A_all: np.ndarray,
    activity_to_idx: Dict[str, int],
    csv_path: Path,
) -> None:
    idx_to_activity = {idx: activity for activity, idx in activity_to_idx.items()}

    rows = []

    for idx in range(A_all.shape[0]):
        eigvals = np.linalg.eigvals(A_all[idx])
        rows.append({
            "activity_encoded": int(idx),
            "activity_label": idx_to_activity.get(idx, str(idx)),
            "spectral_radius": float(np.max(np.abs(eigvals))),
            "max_real_eigenvalue": float(np.max(eigvals.real)),
            "min_real_eigenvalue": float(np.min(eigvals.real)),
        })

    pd.DataFrame(rows).to_csv(csv_path, index=False)


def plot_activity_koopman_eigenvalues(
    A_all: np.ndarray,
    activity_to_idx: Dict[str, int],
    figure_path: Path,
) -> None:
    idx_to_activity = {idx: activity for activity, idx in activity_to_idx.items()}

    theta = np.linspace(0, 2 * np.pi, 400)
    unit_x = np.cos(theta)
    unit_y = np.sin(theta)

    plt.figure(figsize=(8, 8))
    plt.plot(unit_x, unit_y, linestyle="--", label="unit circle")

    for idx in range(A_all.shape[0]):
        eigvals = np.linalg.eigvals(A_all[idx])
        label = f"act {idx_to_activity.get(idx, idx)}"
        plt.scatter(eigvals.real, eigvals.imag, s=16, alpha=0.7, label=label)

    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlabel("Real")
    plt.ylabel("Imaginary")
    plt.title("Activity-conditioned Koopman eigenvalues")
    plt.legend(fontsize=8, ncol=2)
    plt.axis("equal")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


# ============================================================
# Main training pipeline
# ============================================================

def run_kovae_training(config: KoVAEConfig) -> Dict[str, object]:
    paths = get_project_paths(config)
    create_dirs(paths)
    set_random_seed(config.random_seed)

    logger = setup_logging(paths["logs"] / f"{config.experiment_name}_training.log")
    save_json(asdict(config), paths["configs"] / f"{config.experiment_name}_config.json")

    device = get_device()
    logger.info("Starting multi-branch KoVAE training")
    logger.info("Device: %s", device)
    logger.info("Processed data folder: %s", paths["processed"])

    arrays = load_native_rate_arrays(paths["processed"])
    validate_arrays(arrays, config)

    train_loader, val_loader, activity_to_idx, subject_to_idx, split_indices = create_dataloaders(
        arrays=arrays,
        config=config,
        logger=logger,
    )

    save_json(activity_to_idx, paths["results"] / "activity_mapping.json")
    save_json(subject_to_idx, paths["results"] / "subject_mapping.json")
    np.savez_compressed(
        paths["results"] / "split_indices_used_for_kovae.npz",
        train_indices=split_indices["train_indices"],
        val_indices=split_indices["val_indices"],
        test_indices=split_indices["test_indices"],
    )

    model = MultiBranchKoVAE(
        config=config,
        num_activities=len(activity_to_idx),
        num_subject_tokens=len(subject_to_idx),
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    logger.info("Total parameters: %d", total_params)
    logger.info("Trainable parameters: %d", trainable_params)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(config.use_amp and device.type == "cuda"))

    best_val_loss = math.inf
    best_epoch = -1
    epochs_without_improvement = 0
    history_rows = []
    best_A = None
    best_activity_A = None

    unk_subject_index = subject_to_idx["UNK"]

    for epoch in range(1, config.epochs + 1):
        train_metrics, train_A = run_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            config=config,
            device=device,
            train=True,
            unk_subject_index=unk_subject_index,
        )

        val_metrics, val_A = run_one_epoch(
            model=model,
            loader=val_loader,
            optimizer=None,
            scaler=None,
            config=config,
            device=device,
            train=False,
            unk_subject_index=unk_subject_index,
        )

        row = {"epoch": epoch}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history_rows.append(row)

        logger.info(
            "Epoch %03d | train total %.6f | val total %.6f | train recon %.6f | val recon %.6f | koop %.6f | val act acc %.4f",
            epoch,
            train_metrics["total_loss"],
            val_metrics["total_loss"],
            train_metrics["reconstruction_loss"],
            val_metrics["reconstruction_loss"],
            train_metrics["koopman_loss"],
            val_metrics["activity_accuracy"],
        )

        is_best = val_metrics["total_loss"] < best_val_loss

        if is_best:
            best_val_loss = val_metrics["total_loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_A = val_A
            best_activity_A = model.get_activity_koopman_matrices().detach().cpu().numpy()

            save_checkpoint(
                path=paths["checkpoints"] / f"{config.experiment_name}_best.pt",
                model=model,
                optimizer=optimizer,
                config=config,
                epoch=epoch,
                best_val_loss=best_val_loss,
                activity_to_idx=activity_to_idx,
                subject_to_idx=subject_to_idx,
            )
        else:
            epochs_without_improvement += 1

        save_checkpoint(
            path=paths["checkpoints"] / f"{config.experiment_name}_last.pt",
            model=model,
            optimizer=optimizer,
            config=config,
            epoch=epoch,
            best_val_loss=best_val_loss,
            activity_to_idx=activity_to_idx,
            subject_to_idx=subject_to_idx,
        )

        history_df = pd.DataFrame(history_rows)
        history_df.to_csv(paths["results"] / "training_history.csv", index=False)

        if epochs_without_improvement >= config.patience:
            logger.info("Early stopping at epoch %d", epoch)
            break

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(paths["results"] / "training_history.csv", index=False)

    plot_training_history(
        history_df=history_df,
        figure_path=paths["figures"] / "training_loss_curve.png",
    )

    if best_A is not None:
        np.save(paths["results"] / "best_koopman_matrix.npy", best_A)
        plot_koopman_eigenvalues(
            A=best_A,
            figure_path=paths["figures"] / "koopman_eigenvalues.png",
        )

    if best_activity_A is not None:
        np.save(paths["results"] / "best_activity_koopman_matrices.npy", best_activity_A)
        save_activity_koopman_summary(
            A_all=best_activity_A,
            activity_to_idx=activity_to_idx,
            csv_path=paths["results"] / "activity_koopman_summary.csv",
        )
        plot_activity_koopman_eigenvalues(
            A_all=best_activity_A,
            activity_to_idx=activity_to_idx,
            figure_path=paths["figures"] / "activity_koopman_eigenvalues.png",
        )

    plot_reconstruction_examples(
        model=model,
        loader=val_loader,
        config=config,
        device=device,
        figure_path=paths["figures"] / "reconstruction_examples.png",
    )

    final_metrics = {
        "best_epoch": int(best_epoch),
        "best_val_total_loss": float(best_val_loss),
        "best_val_activity_accuracy": float(history_df.loc[history_df["epoch"] == best_epoch, "val_activity_accuracy"].iloc[0])
            if best_epoch > 0 and "val_activity_accuracy" in history_df.columns else None,
        "total_parameters": int(total_params),
        "trainable_parameters": int(trainable_params),
        "num_train_windows": int(len(split_indices["train_indices"])),
        "num_val_windows": int(len(split_indices["val_indices"])),
        "num_test_windows_reserved": int(len(split_indices["test_indices"])),
        "checkpoint_best": str(paths["checkpoints"] / f"{config.experiment_name}_best.pt"),
        "checkpoint_last": str(paths["checkpoints"] / f"{config.experiment_name}_last.pt"),
    }

    save_json(final_metrics, paths["results"] / "final_metrics.json")

    model_summary = {
        "model_name": "ConditionalMultiBranchKoVAE",
        "architecture": {
            "branches": ["BVP", "ACC", "SLOW"],
            "latent_steps": config.latent_steps,
            "latent_dim": config.latent_dim,
            "branch_hidden_dim": config.branch_hidden_dim,
            "fusion_hidden_dim": config.fusion_hidden_dim,
            "conditioned_on_activity": True,
            "conditioned_on_subject": config.use_subject_condition,
            "activity_classifier_head": True,
            "activity_conditioned_koopman": config.use_activity_conditioned_koopman,
        },
        "loss": {
            "reconstruction": "balanced MSE over BVP, ACC, and SLOW branches",
            "kl": "standard VAE KL term",
            "koopman": "activity-conditioned learned latent transition matrices",
            "activity_classifier": "cross-entropy activity supervision on pooled latent mean",
            "alpha_koopman": config.alpha_koopman,
            "beta_kl": config.beta_kl,
            "activity_classifier_weight": config.activity_classifier_weight,
        },
    }

    save_json(model_summary, paths["generators"] / f"{config.experiment_name}_model_summary.json")

    logger.info("Training finished.")
    logger.info("Best epoch: %s", best_epoch)
    logger.info("Best val loss: %.6f", best_val_loss)
    logger.info("Saved outputs in: %s", paths["results"])

    return {
        "final_metrics": final_metrics,
        "paths": {key: str(value) for key, value in paths.items()},
    }


## Run KoVAE training

Run this after Notebook 01 preprocessing has finished.

The important new metrics in `training_history.csv` are:

```text
train_activity_classifier_loss
val_activity_classifier_loss
train_activity_accuracy
val_activity_accuracy
koopman_prediction_loss
```

The new activity-specific Koopman matrices are saved as:

```text
results/kovae/best_activity_koopman_matrices.npy
results/kovae/activity_koopman_summary.csv
figures/kovae/activity_koopman_eigenvalues.png
```


In [2]:
outputs = run_kovae_training(CONFIG)
outputs["final_metrics"]


2026-07-07 17:06:01 | INFO | Starting multi-branch KoVAE training
2026-07-07 17:06:01 | INFO | Device: cuda
2026-07-07 17:06:01 | INFO | Processed data folder: /home/iailab42/khans1/projects/ir/data/processed/native_rates
2026-07-07 17:06:01 | INFO | Activity mapping: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7}
2026-07-07 17:06:01 | INFO | Subject mapping: {'UNK': 0, 'S1': 1, 'S2': 2, 'S3': 3, 'S4': 4, 'S5': 5, 'S6': 6, 'S9': 7, 'S11': 8, 'S12': 9, 'S13': 10}
2026-07-07 17:06:01 | INFO | Train windows: 30762
2026-07-07 17:06:01 | INFO | Val windows: 6100
2026-07-07 17:06:01 | INFO | Test windows reserved for downstream only: 10063
2026-07-07 17:06:01 | INFO | Total parameters: 358334
2026-07-07 17:06:01 | INFO | Trainable parameters: 358334
2026-07-07 17:06:11 | INFO | Epoch 001 | train total 0.457394 | val total 0.203691 | train recon 0.395218 | val recon 0.193925 | koop 0.115555 | val act acc 0.9995
2026-07-07 17:06:19 | INFO | Epoch 002 | train total 0.236683 | 

{'best_epoch': 52,
 'best_val_total_loss': 0.06505395084985945,
 'best_val_activity_accuracy': 1.0,
 'total_parameters': 358334,
 'trainable_parameters': 358334,
 'num_train_windows': 30762,
 'num_val_windows': 6100,
 'num_test_windows_reserved': 10063,
 'checkpoint_best': '/home/iailab42/khans1/projects/ir/models/checkpoints/kovae_best.pt',
 'checkpoint_last': '/home/iailab42/khans1/projects/ir/models/checkpoints/kovae_last.pt'}

## Inspect activity-conditioned Koopman matrices

This checks the spectral radius of every learned activity-specific Koopman matrix.

A value above 1 does not automatically mean training failed, but very large values can make long latent rollouts unstable.


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(CONFIG.project_root)
A_all = np.load(PROJECT_ROOT / "results" / CONFIG.experiment_name / "best_activity_koopman_matrices.npy")

summary_rows = []

for idx, A in enumerate(A_all):
    eigvals = np.linalg.eigvals(A)
    spectral_radius = float(np.max(np.abs(eigvals)))
    summary_rows.append({
        "activity_encoded": idx,
        "spectral_radius": spectral_radius,
        "max_eigenvalue": eigvals[np.argmax(np.abs(eigvals))],
    })

pd.DataFrame(summary_rows)


,activity_encoded,spectral_radius,max_eigenvalue
0,0,1.451715,1.451715+0.000000j
1,1,1.393109,1.393109+0.000000j
2,2,1.279008,1.279008+0.000000j
3,3,1.209046,1.209046+0.000000j
4,4,1.287562,1.287562+0.000000j
5,5,1.408595,1.408595+0.000000j
6,6,1.417343,1.417343+0.000000j
7,7,1.455437,1.455437+0.000000j
